# 00 — Whitelist

**Fase 0 del proyecto ISO 20022 en Redes DLT.**

#TODO: agregar descripcion

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de archivos

# 2. Imports

In [ ]:
import os
import pandas as pd
import httpx

from google.colab import drive
drive.mount('/content/drive')

# 3. Paths y Constantes

In [ ]:
# Paths de salida
BASE_DIR: str = "/content/drive/MyDrive/Tesis/main"
DIR_RAW: dict[str, str] = {
    "stellar": os.path.join(BASE_DIR, "data", "raw", "stellar"),
    "hedera": os.path.join(BASE_DIR, "data", "raw", "hedera"),
    "algorand": os.path.join(BASE_DIR, "data", "raw", "algorand"),}
DIR_EXTERNAL: str = os.path.join(BASE_DIR, "data", "external")

In [ ]:
# Límites de seguridad para las llamadas 
TIMEOUT: int = 30

# Tags que consideramos institucionales (whitelist)
TAGS_WHITELIST: list[str] = ["anchor", "issuer", "exchange", "custodian"]

# Tags de exclusión directa (blacklist)
TAGS_BLACKLIST: list[str] = ["malicious", "unsafe", "airdrop"]

In [ ]:
# Endpoints
STELLAR_DIRECTORY: str = "https://api.stellar.expert/explorer/directory"
XRPSCAN_WELLKNOWN: str = "https://api.xrpscan.com/api/v1/names/well-known"

# Test de conexión

# 4. Funciones

In [ ]:
def stellar_whitelist(tags: list[str], max_paginas: int = 20) -> pd.DataFrame:
    """
    Trae cuentas etiquetadas del directorio de Stellar.Expert.
    Args:
        tags: lista de tags a incluir (p. ej. anchor, issuer).
        max_paginas: tope de páginas a recorrer.
    Returns:
        DataFrame con address, name, domain, tags.
    """
    params = [("tag[]", t) for t in tags]
    params += [("limit", 200), ("order", "asc")]
    filas: list[dict] = []
    url = STELLAR_DIRECTORY
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for pagina in range(max_paginas):
            try:
                r = cliente.get(url, params=params)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error directorio: {e}")
                break
            registros = datos.get("_embedded", {}).get("records", [])
            if not registros:
                break
            for rec in registros:
                filas.append({
                    "address": rec.get("address", ""),
                    "name": rec.get("name", ""),
                    "domain": rec.get("domain", ""),
                    "tags": ",".join(rec.get("tags", [])),
                })
            siguiente = datos.get("_links", {}).get("next", {}).get("href", "")
            if not siguiente:
                break
            url = "https://api.stellar.expert" + siguiente
            params = None  
            if DEBUG:
                print(f"[DEBUG] Stellar dir pág. {pagina + 1} "
                      f"- {len(filas)} cuentas")
    if DEBUG:
        print(f"[DEBUG] Stellar whitelist: {len(filas)} cuentas")
    return pd.DataFrame(filas)


def limpiar_whitelist(df: pd.DataFrame, blacklist: list[str]) -> pd.DataFrame:
    """Excluye cuentas con algún tag de blacklist."""
    def es_limpia(tags_str: str) -> bool:
        tags = set(tags_str.split(","))
        return not (tags & set(blacklist))
    return df[df["tags"].apply(es_limpia)].reset_index(drop=True)

In [ ]:
def xrp_whitelist() -> pd.DataFrame:
    """
    Trae las cuentas conocidas del directorio de XRPScan.
    Returns:
        DataFrame con account, name, domain, verified.
    """
    filas: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        try:
            r = cliente.get(XRPSCAN_WELLKNOWN)
            r.raise_for_status()
            datos = r.json()
        except (httpx.HTTPError, ValueError) as e:
            if DEBUG:
                print(f"[DEBUG] error XRPScan: {e}")
            return pd.DataFrame()
    for rec in datos:
        filas.append({
            "address": rec.get("account", ""),
            "name": rec.get("name", ""),
            "domain": rec.get("domain", ""),
            "verified": rec.get("verified", False),
        })
    if DEBUG:
        print(f"[DEBUG] XRP whitelist: {len(filas)} cuentas")
    return pd.DataFrame(filas)

# 5. Proceso

In [ ]:
wl_stellar = stellar_whitelist(TAGS_WHITELIST)
wl_stellar = limpiar_whitelist(wl_stellar, TAGS_BLACKLIST)
if DEBUG:
  print(wl_stellar["tags"].value_counts())

# 6. Save